# Project Baseline: cohort feasibility in R

This notebook shows how to go from a research question to an analysis-ready cohort in four steps,
using the [Project Baseline Health Study](https://www.projectbaseline.com/) (Limited Tier) data collection.

**Question:** *If we ran a study in young adults with asthma who use an asthma or airway medication,
how many participants would be eligible, and how do they compare with everyone else?*

1. Define eligibility as one SQL query, one criterion per step
2. Look at the eligibility funnel
3. Compare eligible participants with everyone else (Table 1)
4. Save the cohort to a workspace bucket

This is the R version of `cohort_feasibility_python.ipynb`. It runs the same query and gives the same
numbers. Run it in JupyterLab with the R kernel, or in the R Analysis app.

**Requirements:** a workspace with the Project Baseline Health Study (Limited Tier) data collection
added, which creates the `screener`, `crf`, `analysis`, and `externallab` BigQuery resources. To save
results, the workspace also needs a bucket resource (named `uploads` below).

*The eligibility criteria are hypothetical and for illustration only.*

## Setup

In [ ]:
suppressPackageStartupMessages({
  library(bigrquery)
  library(dplyr)
  library(ggplot2)
})

# Return the cloud path of a workspace resource, e.g. "crf" -> "my-project.crf"
resolve <- function(resource_name) {
  system2("wb", c("resolve", "--name", resource_name), stdout = TRUE, stderr = FALSE)
}

SCREENER <- resolve("screener")
CRF <- resolve("crf")
ANALYSIS <- resolve("analysis")
EXTERNALLAB <- resolve("externallab")
OUTPUT_BUCKET_RESOURCE <- "uploads"  # change to any bucket resource in your workspace
BILLING_PROJECT <- Sys.getenv("GOOGLE_CLOUD_PROJECT")
CRF

## Step 1: Define eligibility in one query

Each `WITH` block is one criterion, so the logic is easy to read, audit, and change. The final
`SELECT` returns one row per participant with a true/false column per criterion, plus a few
characteristics for Table 1.

Tip: in `crf.CM`, a drug that belongs to several ATC classes lists all of them in one field
(for example `G02CA:... || R03AC:...`), so match codes with `REGEXP_CONTAINS` rather than `=`.

In [ ]:
cohort_sql <- glue::glue("
WITH
demographics AS (
  SELECT USUBJID, SAFE_CAST(age_at_enrollment AS INT64) AS age, SEX AS sex, RACE AS race
  FROM `{SCREENER}.DM`),
asthma AS (        -- medical history includes asthma
  SELECT DISTINCT USUBJID FROM `{CRF}.MH` WHERE PTName = 'Asthma'),
airway_med AS (    -- any drug for obstructive airway disease (ATC R03), relievers and controllers
  SELECT DISTINCT USUBJID FROM `{CRF}.CM` WHERE REGEXP_CONTAINS(ATC_Codes, 'R03')),
copd AS (          -- medical history includes COPD
  SELECT DISTINCT USUBJID FROM `{CRF}.MH` WHERE PTName = 'Chronic obstructive pulmonary disease'),
retained AS (      -- attended at least 3 annual visits
  SELECT USUBJID FROM `{CRF}.VISDT1`
  WHERE VISIT LIKE '%ANNUAL_VISIT' GROUP BY USUBJID HAVING COUNT(DISTINCT VISIT) >= 3),
bmi AS (
  SELECT USUBJID, AVG(SAFE_CAST(vs_weight_kg AS FLOAT64) / POW(SAFE_CAST(vs_height_cm AS FLOAT64) / 100, 2)) AS bmi
  FROM `{CRF}.VS` WHERE VISIT = 'STUDY_START' GROUP BY USUBJID),
phq9 AS (          -- depression symptom score, first study year
  SELECT USUBJID, AVG(PHQ9_SUM_SCORE) AS phq9
  FROM `{ANALYSIS}.PHQ9A_SCORES` WHERE VISITNUM < 2 GROUP BY USUBJID),
crp AS (           -- high-sensitivity C-reactive protein at enrollment
  SELECT USUBJID, AVG(SAFE_CAST(siurslt AS FLOAT64)) AS hs_crp
  FROM `{EXTERNALLAB}.CLABS` WHERE VISIT = 'STUDY_START' AND rtcname = 'CRP HIGH SENSITIVITY' GROUP BY USUBJID)
SELECT
  d.*,
  d.USUBJID IN (SELECT USUBJID FROM asthma)         AS has_asthma,
  d.age BETWEEN 18 AND 40                           AS age_18_40,
  d.USUBJID IN (SELECT USUBJID FROM airway_med)     AS on_airway_med,
  d.USUBJID NOT IN (SELECT USUBJID FROM copd)       AS no_copd,
  d.USUBJID IN (SELECT USUBJID FROM retained)       AS retained,
  bmi.bmi, phq9.phq9, crp.hs_crp
FROM demographics d
LEFT JOIN bmi USING (USUBJID)
LEFT JOIN phq9 USING (USUBJID)
LEFT JOIN crp USING (USUBJID)
ORDER BY USUBJID
")
participants <- bq_table_download(bq_project_query(BILLING_PROJECT, cohort_sql))
head(participants)

## Step 2: Eligibility funnel

Apply the criteria one at a time and count who is left after each.

In [ ]:
criteria <- c(
  "Asthma" = "has_asthma",
  "Age 18-40" = "age_18_40",
  "On an airway medication (R03)" = "on_airway_med",
  "No COPD" = "no_copd",
  "3+ annual visits" = "retained"
)
eligible <- rep(TRUE, nrow(participants))
funnel <- tibble(criterion = "All participants", participants = nrow(participants))
for (label in names(criteria)) {
  eligible <- eligible & coalesce(participants[[criteria[[label]]]], FALSE)
  funnel <- add_row(funnel, criterion = label, participants = sum(eligible))
}
participants$eligible <- eligible

options(repr.plot.width = 7, repr.plot.height = 3)
ggplot(funnel, aes(x = participants, y = factor(criterion, levels = rev(criterion)))) +
  geom_col(fill = "steelblue") +
  geom_text(aes(label = participants), hjust = -0.2) +
  scale_x_continuous(expand = expansion(mult = c(0, 0.1))) +
  labs(x = "participants", y = NULL) +
  theme_minimal()
funnel

## Step 3: Table 1, eligible vs. everyone else

In [ ]:
median_iqr <- function(values) {
  q <- quantile(values, c(0.25, 0.5, 0.75), na.rm = TRUE)
  sprintf("%.1f [%.1f-%.1f]", q[2], q[1], q[3])
}

table1 <- participants |>
  mutate(group = if_else(eligible, "Eligible", "Not eligible")) |>
  group_by(group) |>
  summarise(
    "N" = as.character(n()),
    "Age, median [IQR]" = median_iqr(age),
    "Female, %" = as.character(round(100 * mean(sex == "Female"))),
    "Non-White, %" = as.character(round(100 * mean(coalesce(race, "") != "White"))),
    "BMI, median [IQR]" = median_iqr(bmi),
    "PHQ-9, median [IQR]" = median_iqr(phq9),
    "hs-CRP mg/L, median [IQR]" = median_iqr(hs_crp)
  ) |>
  tidyr::pivot_longer(-group, names_to = "characteristic") |>
  tidyr::pivot_wider(names_from = group, values_from = value)
table1

## Step 4: Save the cohort to a workspace bucket

Save the cohort, Table 1, and the exact SQL that defined them, so a collaborator can rerun or audit the
analysis later. Everything stays inside the workspace.

In [ ]:
cohort <- filter(participants, eligible)
write.csv(cohort, "cohort.csv", row.names = FALSE)
write.csv(table1, "table1.csv", row.names = FALSE)
writeLines(cohort_sql, "cohort_definition.sql")

destination <- paste0(resolve(OUTPUT_BUCKET_RESOURCE), "/cohort_feasibility_r/")
system2("gcloud", c("storage", "cp", "cohort.csv", "table1.csv", "cohort_definition.sql", destination),
        stdout = FALSE, stderr = FALSE)
cat("Saved", nrow(cohort), "participants to", destination, "\n")

---
Copyright 2026 Verily Life Sciences LLC

Use of this source code is governed by a BSD-style
license that can be found in the LICENSE file or at
https://developers.google.com/open-source/licenses/bsd